### 문제 1-1 : 기본 Chain 만들기 - AI 요리사


### 문제 설명
#### 사용자가 재료를 입력하면 그 재료로 만들 수 있는 요리를 추천해주는 간단한 AI 요리사를 만들어보세요.


In [3]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

GROQ_BASE_URL = "https://api.groq.com/openai/v1"
GROQ_MODEL = "openai/gpt-oss-120b"
TEMPERATURE = 0.7

def get_llm(model: str = GROQ_MODEL, temperature: float = TEMPERATURE) -> ChatOpenAI:
    """Groq 엔드포인트를 사용하는 ChatOpenAI 인스턴스를 생성합니다."""
    return ChatOpenAI(
        api_key=GROQ_API_KEY,
        base_url=GROQ_BASE_URL,
        model=model,
        temperature=temperature,
    )

llm = get_llm()
print(f"model name={llm.model_name}")

model name=openai/gpt-oss-120b


### 문제 1-1 : 기본 Chain 만들기 - AI 요리사

In [8]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. PromptTemplate: {ingredients} 자리에 사용자가 입력한 재료가 들어감
cook_prompt = PromptTemplate.from_template(
    """당신은 친절한 AI 요리사입니다.
사용자가 가진 재료로 만들 수 있는 요리 1가지를 추천하고 간단한 레시피를 알려주세요.
반드시 아래 형식을 그대로 지켜서 한국어로 답하세요.

[출력 형식]
(재료)로 만들 수 있는 요리를 추천드립니다!

추천 요리: (요리 이름)
재료: (필요한 재료)
조리법:
1. (단계)
2. (단계)
3. (단계)

재료: {ingredients}"""
)

# 2. ChatOpenAI 모델: 셀 0에서 만든 llm 사용 (get_llm → ChatOpenAI)

# 3. StrOutputParser: AIMessage → 문자열
output_parser = StrOutputParser()

# 4. LCEL(|)로 체인 연결
cook_chain = cook_prompt | llm | output_parser

# 5. 실행
ingredients = "계란, 밥, 김치"
# ingredients = input("재료를 입력하세요: ")   # 직접 입력받으려면 이 줄의 주석 해제

result = cook_chain.invoke({"ingredients": ingredients})

print(f'입력: "{ingredients}"')
print("출력:")
print(result)

입력: "계란, 밥, 김치"
출력:
계란, 밥, 김치 로 만들 수 있는 요리를 추천드립니다!

추천 요리: 김치볶음밥
재료: 밥 1공기, 김치 100g, 계란 1개, 식용유 1큰술, 참기름 1작은술, 대파(선택) 약간, 소금·후추 약간
조리법:
1. 팬에 식용유를 두르고 다진 김치를 넣어 중불에서 2~3분 정도 볶아 김치의 풍미를 끓인다.
2. 밥을 넣고 김치와 잘 섞이도록 골고루 볶은 뒤, 소금·후추로 간을 맞추고 마지막에 참기름을 뿌린다.
3. 다른 팬에 기름을 살짝 두르고 계란을 프라이한 뒤, 김치볶음밥 위에 올려 완성한다. (원한다면 대파를 얇게 썰어 곁들여도 좋습니다.)


### 문제 1-2 : 2단계 Multi Chain 만들기 - 여행지 정보 시스템

In [9]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# 1단계 프롬프트: 여행지 → 대표 명소 1가지 (이름만)
attraction_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 여행 전문가입니다. 질문한 여행지의 가장 대표적인 관광 명소 1곳의 이름만 답하세요. "
               "설명, 따옴표, 마침표 없이 명소 이름만 출력하세요."),
    ("user", "{destination}의 대표 관광 명소 1가지를 추천해 주세요."),
])

# 2단계 프롬프트: 명소 → 상세 정보 (역사, 특징, 방문 팁)
detail_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 친절한 여행 가이드입니다. 관광 명소에 대해 정확하고 간결하게 안내하세요."),
    ("user", """{destination}의 '{attraction}'에 대해 아래 형식을 그대로 지켜 알려주세요.

                명소: {attraction}
                역사:
                    1.
                    2.

                특징:
                    1.
                    2.

                방문 팁:
                    1.
                    2.
    """),
])

# 각 단계 체인
attraction_chain = attraction_prompt | llm | StrOutputParser()   # 1단계
detail_chain = detail_prompt | llm | StrOutputParser()           # 2단계

# 두 체인을 LCEL로 연결
# {destination} → +attraction(1단계 결과) → +detail(2단계 결과)
travel_chain = (
    RunnablePassthrough.assign(attraction=attraction_chain)
    | RunnablePassthrough.assign(detail=detail_chain)
)

# 실행 및 단계별 결과 출력
destination = "로마"
result = travel_chain.invoke({"destination": destination})

print(f'입력: "{result["destination"]}"')
print(f'\n🔹 1단계 결과: "{result["attraction"].strip()}"')
print(f"\n🔹 2단계 결과:\n{result['detail']}")

입력: "로마"

🔹 1단계 결과: "콜로세움"

🔹 2단계 결과:
                명소: 콜로세움
                역사:
                    1. 서기 72~80년 사이에 베스파시아누스 황제와 그의 아들 티투스 황제에 의해 건축된 로마 제국의 원형극장으로, 원래는 ‘플라비우스 원형극장(Flavian Amphitheatre)’이라 불렸다.  
                    2. 중세에는 성당, 무기고, 수용소 등으로 전환되었으며, 18세기 이후 보존 운동이 일어나 현재는 세계문화유산(1980)으로 지정돼 매년 수백만 명이 찾는다.

                특징:
                    1. 약 5만 명을 수용할 수 있는 거대한 구조(높이 48 m, 장축 189 m, 단축 156 m)와 80개의 입구, 4층 규모의 복합적인 아치형 구조가 돋보인다.  
                    2. 원형 경기장 바닥은 원래 목재와 모래로 된 ‘아레나( arena)’였으며, 검투사 전투, 야생동물 사냥, 해전 재현 등 다양한 공개 오락이 열렸다.

                방문 팁:
                    1. 입장권은 온라인 사전 예매가 필수이며, ‘오전 9시 개장’ 전이나 ‘마지막 입장 시간(오후 4시)’ 직전 방문하면 인파를 피할 수 있다.  
                    2. 2층·3층 전망대와 지하 ‘하이포가’(검투사와 동물들이 머물던 방) 투어를 선택하면 콜로세움의 구조와 역사를 더 깊이 체험할 수 있다.


### 문제 1-3 : FewShotPromptTemplate과 시스템 메시지 활용

In [10]:
from langchain_core.prompts import FewShotChatMessagePromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. 예시 데이터: 문제 제공 2개 + 추가 1개 = 3개 (IT / 보건 / 스포츠)
examples = [
    {
        "news": "삼성전자가 내년 초에 자체적으로 개발한 인공지능(AI) 가속기를 처음으로 출시할 예정이다. 이는 AI 반도체 시장에서 지배적인 위치를 차지하고 있는 엔비디아의 독점을 도전하고, 세계 최고의 반도체 제조업체로서의 지위를 다시 확립하려는 삼성전자의 노력으로 해석된다.",
        "keywords": "삼성전자, 인공지능, 엔비디아",
    },
    {
        "news": "세계보건기구(WHO)는 최근 새로운 건강 위기에 대응하기 위해 국제 협력의 중요성을 강조했다. 전염병 대응 역량의 강화와 글로벌 보건 시스템의 개선이 필요하다고 발표했다.",
        "keywords": "세계보건기구, 건강위기, 국제협력",
    },
    {   # 추가 예시
        "news": "손흥민이 프리미어리그 경기에서 멀티골을 기록하며 팀의 3-1 승리를 이끌었다. 이로써 그는 이번 시즌 10호 골을 달성하며 팀 내 득점 1위에 올랐다.",
        "keywords": "손흥민, 프리미어리그, 멀티골",
    },
]

# 2. 예시 프롬프트 (대화형)
example_prompt = ChatPromptTemplate.from_messages([
    ("human", "{news}"),
    ("ai", "키워드: {keywords}"),
])

# 3. Few-Shot 프롬프트
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

# 4. 최종 프롬프트: system + 예시 + 실제 입력
final_prompt = ChatPromptTemplate.from_messages([
    ("system", "뉴스 키워드 추출 전문가입니다. 핵심 키워드 3개를 추출하세요. "
               "예시와 동일하게 '키워드: A, B, C' 형식으로만 답하고 설명은 붙이지 마세요."),
    few_shot_prompt,
    ("human", "{input}"),
])

# 5. 체인 연결 (LCEL)
keyword_chain = final_prompt | llm | StrOutputParser()

# 6. 다양한 분야 테스트
test_news_list = [
    # IT (문제 제공 테스트 뉴스)
    "제미나이 2.0 플래시는 현재 구글 AI 스튜디오(Google AI Studio) 및 버텍스 AI(Vertex AI)에서 제미나이 API를 통해 개발자에게 실험 모델로 제공됩니다. 모든 개발자는 멀티모달 입력 및 텍스트 출력을 사용할 수 있으며, 텍스트 음성 변환(text-to-speech) 및 네이티브 이미지 생성은 일부 파트너들을 대상으로 제공됩니다. 내년 1월에는 더 많은 모델 사이즈와 함께 일반에 공개될 예정입니다.",
    # 경제
    "한국은행 금융통화위원회는 기준금리를 연 3.25%로 동결했다. 물가 상승률은 둔화됐지만 가계부채 증가세와 환율 변동성을 고려한 결정이다.",
    # 날씨
    "기상청은 올겨울 평년보다 기온이 낮고 한파가 잦을 것으로 전망했다. 특히 12월 중순에는 강한 추위가 예상된다.",
]

for news in test_news_list:
    result = keyword_chain.invoke({"input": news})
    print(f"뉴스: {news[:40]}...")
    print(result)
    print("-" * 50)

뉴스: 제미나이 2.0 플래시는 현재 구글 AI 스튜디오(Google AI St...
키워드: 제미나이, 구글 AI 스튜디오, 멀티모달
--------------------------------------------------
뉴스: 한국은행 금융통화위원회는 기준금리를 연 3.25%로 동결했다. 물가 상승...
키워드: 한국은행, 기준금리, 가계부채
--------------------------------------------------
뉴스: 기상청은 올겨울 평년보다 기온이 낮고 한파가 잦을 것으로 전망했다. 특히...
키워드: 기상청, 겨울, 한파
--------------------------------------------------
